# Step 2 (Stage 2.1): Managed Spark Bronze Load & PII Pseudonymisation (`@skill:gcp-spark` & `@skill:biglake-management`)
**Complementary 2-Stage Pipeline Architecture & Dual-Catalog Lakehouse (`lakehouse-demo` Runtime 2.3 Pattern):**
- **Dataproc Serverless Runtime `2.3` (Spark 3.5 + Iceberg `1.5.0`)**: Uses the exact `lakehouse-demo` session template (`clarity-lrc-spark-template`, Runtime `2.3`, `iceberg-spark-runtime-3.5_2.13:1.5.0`, `iceberg-gcp-bundle:1.5.0`, `delta-spark_2.13:3.2.0`), supporting both the **VS Code DAK Remote Serverless Spark Kernel (`jupyterSession`)** and **Local Python `DataprocSparkSession`**.
- **Catalog 1 (`biglakefederated` — Databricks Unity Catalog Federation)**: Hosts the single cross-cloud raw dataset (`biglakefederated.raw_search_telemetry.search_events` backed by `gs://your-gcp-project-id-iceberg-dbx-lakehouse`).
- **Catalog 2 (`lrc` — GCP-Native BigLake Iceberg REST Catalog)**: Hosts GCP-native Lakehouse tables (`lrc.modeled_search_warehouse.sessionized_searches`) backed by `gs://your-gcp-project-id-iceberg-lakehouse` with `CREDENTIAL_MODE_VENDED_CREDENTIALS` (`X-Iceberg-Access-Delegation=vended-credentials`).
- **Stage 2.1 (Serverless Spark)**: Reads `biglakefederated.raw_search_telemetry.search_events` (with automatic fallback to `gs://your-gcp-project-id-iceberg-lakehouse/dbx_seed_data/search_events.jsonl`), applies **salted SHA-256 PII pseudonymisation** (`pseudonymized_customer_id`), filters scraper bots, computes 30-minute session boundaries, and writes the governed Silver Iceberg table `lrc.modeled_search_warehouse.sessionized_searches` using **Git-style Write-Audit-Publish (WAP) branching** and **Zero-Scan Metadata Rollbacks**.
- **Stage 2.2 (BigQuery SQL via `dbt-bigquery` & DEA)**: Queries `lrc.modeled_search_warehouse.sessionized_searches` directly in-place (zero data movement) and joins with native BigQuery tables (`search_impressions`, `catalog_items`, `customer_clv`) to build the Gold conversion mart (`fct_search_conversions`).

In [ ]:
import os
import google.auth
import google.auth.transport.requests
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
SESSION_TEMPLATE = "clarity-lrc-spark-template"
WAREHOUSE_BUCKET = f"gs://{PROJECT_ID}-iceberg-lakehouse"

# Refresh ADC credentials and construct Bearer token for Iceberg 1.5.0 REST Catalogs
# (Exact pattern from lakehouse-demo/adtech-demo & telco-demo on Dataproc Serverless Runtime 2.3)
_creds, _ = google.auth.default()
_creds.refresh(google.auth.transport.requests.Request())
_auth_header = "Bearer " + _creds.token

if os.path.exists("/usr/lib/spark"):
    # Connected directly to Remote Dataproc Serverless Jupyter Kernel (Runtime 2.3 via VS Code DAK Kernel Picker)
    spark = (
        SparkSession.builder.appName("clarity-uc4-interactive-spark-medallion")
        .config("spark.sql.catalog.lrc", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lrc.type", "rest")
        .config("spark.sql.catalog.lrc.uri", "https://biglake.googleapis.com/iceberg/v1/restcatalog")
        .config("spark.sql.catalog.lrc.warehouse", WAREHOUSE_BUCKET)
        .config("spark.sql.catalog.lrc.header.x-goog-user-project", PROJECT_ID)
        .config("spark.sql.catalog.lrc.header.X-Iceberg-Access-Delegation", "vended-credentials")
        .config("spark.sql.catalog.lrc.header.Authorization", _auth_header)
        .config("spark.sql.catalog.lrc.io-impl", "org.apache.iceberg.gcp.gcs.GCSFileIO")
        .config("spark.sql.catalog.biglakefederated", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.biglakefederated.type", "rest")
        .config("spark.sql.catalog.biglakefederated.uri", "https://biglake.googleapis.com/iceberg/v1/restcatalog")
        .config("spark.sql.catalog.biglakefederated.warehouse", f"bl://projects/{PROJECT_ID}/catalogs/biglakefederated")
        .config("spark.sql.catalog.biglakefederated.header.x-goog-user-project", PROJECT_ID)
        .config("spark.sql.catalog.biglakefederated.header.X-Iceberg-Access-Delegation", "vended-credentials")
        .config("spark.sql.catalog.biglakefederated.header.Authorization", _auth_header)
        .config("spark.sql.catalog.biglakefederated.io-impl", "org.apache.iceberg.gcp.gcs.GCSFileIO")
        .getOrCreate()
    )
    for _cat in ("lrc", "biglakefederated"):
        spark.conf.set(f"spark.sql.catalog.{_cat}.header.Authorization", _auth_header)
else:
    # Connected to Local Python Kernel -> Use DataprocSparkSession with Runtime 2.3 Session Template
    from google.cloud.dataproc_spark_connect import DataprocSparkSession
    from google.cloud.dataproc_v1 import Session

    _session = Session()
    _session.session_template = f"projects/{PROJECT_ID}/locations/{REGION}/sessionTemplates/{SESSION_TEMPLATE}"
    for _cat in ("lrc", "biglakefederated"):
        _session.runtime_config.properties[f"spark.sql.catalog.{_cat}.header.Authorization"] = _auth_header

    spark = (
        DataprocSparkSession.builder.appName("clarity-uc4-interactive-spark-medallion")
        .dataprocSessionConfig(_session)
        .getOrCreate()
    )

print(f"Spark {spark.version} ready — Dual-Catalog: GCP Native LRC (`lrc`) + DBX Federation (`biglakefederated`)")


def read_cross_cloud_search_events(
    spark_session: SparkSession,
    federated_table: str = "biglakefederated.raw_search_telemetry.search_events",
    seed_fallback_uri: str = f"gs://{PROJECT_ID}-iceberg-lakehouse/dbx_seed_data/search_events.jsonl",
) -> DataFrame:
    """Read live Databricks federated table (`biglakefederated.raw_search_telemetry.search_events`)
    with automatic fallback to GCS seed JSONL.
    """
    try:
        df = spark_session.read.table(federated_table)
        df.limit(1).count()
        print(f"[Dual-Catalog LRC] Reading live DBX federated table: {federated_table}")
        return df
    except Exception as exc:
        print(f"[Dual-Catalog LRC] Federated table fallback ({exc}) -> reading {seed_fallback_uri}")
        return spark_session.read.json(seed_fallback_uri)


bronze_df = read_cross_cloud_search_events(spark)
# Mandatory @skill:gcp-spark rule: printSchema before transformation
bronze_df.printSchema()

### Stage 2.1 Transformation: SHA-256 PII Pseudonymisation, Bot Filtering, and 30-Minute Session Windows
We hash customer identifiers with salted SHA-256 (`pseudonymized_customer_id`), remove `PriceScraperBot` rows, deduplicate network retry events, and compute 30-minute (`1800s`) inactivity session boundaries before writing to the **GCP-Native BigLake Iceberg catalog (`lrc.modeled_search_warehouse.sessionized_searches`)** for downstream `dbt-bigquery` ELT models.

In [ ]:
def pseudonymize_pii_columns(df: DataFrame, pii_salt: str = "dak-enterprise-pii-salt-v1") -> DataFrame:
    """Apply cryptographic SHA-256 pseudonymisation to customer PII identifiers at the Bronze boundary."""
    existing_cols = set(df.columns)
    result_df = df.withColumn(
        "pseudonymized_customer_id",
        F.sha2(F.concat_ws(":", F.lit(pii_salt), F.trim(F.col("customer_id"))), 256),
    )
    if "user_email" in existing_cols:
        result_df = result_df.withColumn(
            "pseudonymized_email_hash",
            F.sha2(F.concat_ws(":", F.lit(pii_salt), F.lower(F.trim(F.col("user_email")))), 256),
        ).drop("user_email")
    if "ip_address" in existing_cols:
        result_df = result_df.withColumn(
            "pseudonymized_ip_hash",
            F.sha2(F.concat_ws(":", F.lit(pii_salt), F.trim(F.col("ip_address"))), 256),
        ).drop("ip_address")
    if "client_ip" in existing_cols:
        result_df = result_df.withColumn(
            "pseudonymized_ip_hash",
            F.sha2(F.concat_ws(":", F.lit(pii_salt), F.trim(F.col("client_ip"))), 256),
        ).drop("client_ip")
    return result_df


def transform_bronze_to_silver(raw_events_df: DataFrame, pii_salt: str = "dak-enterprise-pii-salt-v1") -> DataFrame:
    """Pseudonymise PII, clean raw search events, filter bots, deduplicate, and sessionize with 30-min windows."""
    raw_events_df.printSchema()
    pseudonymized_df = pseudonymize_pii_columns(raw_events_df, pii_salt=pii_salt)

    cleaned_df = (
        pseudonymized_df.filter(
            ~F.lower(F.coalesce(F.col("user_agent"), F.lit(""))).rlike("scraperbot|crawler|spider")
        )
        .withColumn("search_event_id", F.trim(F.col("search_event_id")))
        .withColumn("customer_id", F.trim(F.col("customer_id")))
        .withColumn("normalized_query_text", F.lower(F.trim(F.col("raw_query_text"))))
        .withColumn("device_type", F.lower(F.coalesce(F.trim(F.col("device_type")), F.lit("unknown_device"))))
        .withColumn("event_ts", F.to_timestamp(F.col("event_timestamp")))
        .withColumn("event_date", F.to_date(F.col("event_date")))
    )

    dedup_window = Window.partitionBy("search_event_id").orderBy(F.col("event_ts").desc())
    dedup_df = (
        cleaned_df.withColumn("_row_num", F.row_number().over(dedup_window))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )

    cust_window = Window.partitionBy("pseudonymized_customer_id").orderBy("event_ts")
    with_lag_df = dedup_df.withColumn("prev_event_ts", F.lag("event_ts").over(cust_window)).withColumn(
        "is_new_session",
        F.when(
            F.col("prev_event_ts").isNull()
            | ((F.unix_timestamp("event_ts") - F.unix_timestamp("prev_event_ts")) > 1800),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )

    sessionized_df = with_lag_df.withColumn(
        "session_seq_num", F.sum("is_new_session").over(cust_window)
    ).withColumn(
        "clarity_session_id",
        F.concat_ws(
            "-S",
            F.substring(F.col("pseudonymized_customer_id"), 1, 12),
            F.lpad(F.col("session_seq_num").cast("string"), 3, "0"),
        ),
    )

    return sessionized_df.drop("prev_event_ts", "is_new_session", "session_seq_num")


silver_df = transform_bronze_to_silver(bronze_df)
silver_df.printSchema()

spark.sql("CREATE NAMESPACE IF NOT EXISTS lrc.modeled_search_warehouse")
spark.sql("DROP TABLE IF EXISTS lrc.modeled_search_warehouse.sessionized_searches")
(
    silver_df.writeTo("lrc.modeled_search_warehouse.sessionized_searches")
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .tableProperty("write.wap.enabled", "true")
    .partitionedBy("event_date")
    .createOrReplace()
)
print(f"✓ lrc.modeled_search_warehouse.sessionized_searches: {spark.table('lrc.modeled_search_warehouse.sessionized_searches').count():,d} rows")

### Git-Style Lakehouse Table Branching (WAP) & Zero-Scan Rollbacks in Interactive Serverless Spark
Using `DataprocSparkSession` over GCP-Native BigLake Managed Iceberg (`lrc`):
1. **Git-Style Branching (Write-Audit-Publish)**: Create an isolated branch (`ALTER TABLE ... CREATE BRANCH IF NOT EXISTS audit_branch`), write staged data to `.branch_audit_branch`, validate data quality rules without mutating `main`, and fast-forward merge (`CALL lrc.system.fast_forward(...)`).
2. **Zero-Scan Metadata Rollback**: Inspect `.snapshots` lineage and execute `CALL lrc.system.rollback_to_snapshot(...)` to instantaneously revert table state via metadata pointer update with **zero Parquet file scan or rewrite**.

In [ ]:
import google.auth
import google.auth.transport.requests
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

# Ensure spark session and fresh Bearer token are active (even if kernel was restarted)
_creds, _ = google.auth.default()
_creds.refresh(google.auth.transport.requests.Request())
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.catalog.lrc.header.Authorization", "Bearer " + _creds.token)


def write_to_iceberg_branch_and_audit(
    spark_session: SparkSession,
    df_to_write: DataFrame,
    table_ident: str = "lrc.modeled_search_warehouse.sessionized_searches",
    branch_name: str = "audit_branch",
) -> bool:
    """Demonstrate Git-style Lakehouse table branching (Write-Audit-Publish / WAP) on BigLake Iceberg."""
    spark_session.sql(f"ALTER TABLE {table_ident} CREATE BRANCH IF NOT EXISTS {branch_name} RETAIN 7 DAYS")
    df_to_write.writeTo(f"{table_ident}.branch_{branch_name}").append()

    branch_df = spark_session.read.table(f"{table_ident}.branch_{branch_name}")
    invalid_rows = branch_df.filter(
        F.col("pseudonymized_customer_id").isNull() | F.col("clarity_session_id").isNull()
    ).count()

    if invalid_rows == 0:
        spark_session.sql(
            f"CALL lrc.system.fast_forward('{table_ident.removeprefix('lrc.')}', 'main', '{branch_name}')"
        )
        return True
    return False


def rollback_iceberg_table_zero_scan(
    spark_session: SparkSession,
    table_ident: str = "lrc.modeled_search_warehouse.sessionized_searches",
    snapshot_id: int | None = None,
) -> int:
    """Execute an instant zero-scan metadata rollback on a BigLake Iceberg Lakehouse table."""
    snapshots_df = spark_session.read.format("iceberg").load(f"{table_ident}.snapshots")
    if snapshot_id is None:
        latest_two = (
            snapshots_df.orderBy(F.col("committed_at").desc())
            .select("snapshot_id")
            .limit(2)
            .collect()
        )
        target_snapshot_id = int(latest_two[-1]["snapshot_id"])
    else:
        target_snapshot_id = int(snapshot_id)

    catalog_relative_table = table_ident.removeprefix("lrc.")
    spark_session.sql(f"CALL lrc.system.rollback_to_snapshot('{catalog_relative_table}', {target_snapshot_id})")
    return target_snapshot_id


# Use a 50-row micro-batch from the already-materialized table so WAP branch append runs in ~2s
# without recomputing the full Bronze->Silver window shuffle:
audit_batch_df = spark.table("lrc.modeled_search_warehouse.sessionized_searches").limit(50)

# 1. Write-Audit-Publish (WAP) on isolated Iceberg branch `audit_branch`
branch_passed = write_to_iceberg_branch_and_audit(
    spark,
    audit_batch_df,
    table_ident="lrc.modeled_search_warehouse.sessionized_searches",
    branch_name="audit_branch",
)
print(f"Git-style Iceberg WAP branch audit passed & fast-forwarded: {branch_passed}")

# 2. Inspect Iceberg snapshot history and demonstrate zero-scan metadata rollback
spark.read.format("iceberg").load("lrc.modeled_search_warehouse.sessionized_searches.snapshots").show(5, truncate=False)
restored_snapshot = rollback_iceberg_table_zero_scan(
    spark,
    table_ident="lrc.modeled_search_warehouse.sessionized_searches",
)
print(f"Executed instant zero-scan metadata rollback to snapshot_id={restored_snapshot}")